# Predicción de clientes en riesgo de abandono

### Algoritmo XGBClassifier

### Descripción del problema
El objetivo de este cuaderno es desarrollar un modelo predictivo de abandono de clientes (churn prediction) para identificar con anticipación a aquellos clientes que presentan un alto riesgo de dejar de comprar en la empresa.

A partir de datos transaccionales y de comportamiento obtenidos directamente de una base de datos SQL Server (WideWorldImporters) —incluyendo información como fecha de apertura de cuenta, estado de crédito, número de compras, monto total adquirido y días desde la última compra— se construye un conjunto de características (features) que permiten modelar el perfil de clientes.

### El problema se aborda como una tarea de clasificación binaria, donde:

Clase 1 (Cliente en riesgo): clientes que cumplen con criterios históricos de inactividad o bajo volumen de compra.

Clase 0 (Cliente activo): clientes que mantienen un comportamiento de compra saludable.

Se utiliza el algoritmo XGBClassifier debido a su capacidad para manejar datos heterogéneos, capturar relaciones no lineales y manejar desbalance de clases. El flujo incluye:

Extracción de datos desde SQL Server.

Preprocesamiento (limpieza, tratamiento de valores nulos, conversión de tipos de datos).

Creación de variable objetivo (ClienteEnRiesgo) con reglas de negocio.

Entrenamiento y validación de un modelo de machine learning.

Evaluación con métricas de clasificación para medir la capacidad predictiva del modelo.



In [3]:
#!pip install pyodbc pandas 

In [4]:
import pyodbc
import pandas as pd
import warnings 
warnings.filterwarnings("ignore")

In [5]:
# Datos de conexion
server = "202102708-VOLX\SQLEXPRESS" # Nombre de la instancia
database ="WideWorldImporters"
username = "sa"
password ="123456"

# cadena de conexion 
conn_str =(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password}"
)

#Crear conexión
try:
    conn = pyodbc.connect(conn_str)
    print("Conexión exitosa a SQL Server")
except Exception as e:
    print("Error al conectar: ", e)

Conexión exitosa a SQL Server


In [6]:
# Consulta directa a SQL
query ="""
SELECT 
    c.CustomerID,
    c.CustomerName,
    c.AccountOpenedDate,
    c.IsOnCreditHold,
    c.CustomerCategoryID,
    c.CreditLimit,
    c.DeliveryMethodID,
    MAX(i.InvoiceDate) AS UltimaCompra,
    DATEDIFF(DAY, MAX(i.InvoiceDate), GETDATE()) AS DiasDesdeUltimaCompra,
    COUNT(DISTINCT i.InvoiceID) AS NumFacturas,
    SUM(il.ExtendedPrice + il.TaxAmount) AS TotalComprado,
    CASE 
        WHEN 
            DATEDIFF(DAY, MAX(i.InvoiceDate), GETDATE()) > 1000 
            AND COUNT(DISTINCT i.InvoiceID) BETWEEN 10 AND 100
            AND SUM(il.ExtendedPrice) < 150000
        THEN 1
        ELSE 0
    END AS ClienteEnRiesgo
FROM Sales.Customers c
LEFT JOIN Sales.Invoices i ON c.CustomerID = i.CustomerID
LEFT JOIN Sales.InvoiceLines il ON i.InvoiceID = il.InvoiceID
GROUP BY 
    c.CustomerID,
    c.CustomerName,
    c.AccountOpenedDate,
    c.IsOnCreditHold,
    c.CustomerCategoryID,
    c.CreditLimit,
    c.DeliveryMethodID
"""


# Leer el resultado en DataFrame
df_churn = pd.read_sql(query, conn)

### Exploracion inicial 

In [8]:
df_churn.head()

,CustomerID,CustomerName,AccountOpenedDate,IsOnCreditHold,CustomerCategoryID,CreditLimit,DeliveryMethodID,UltimaCompra,DiasDesdeUltimaCompra,NumFacturas,TotalComprado,ClienteEnRiesgo
0,1,Tailspin Toys (Head Office),2013-01-01,False,3,NaN,3,2016-05-27,3363,123,397101.76,0
1,2,"Tailspin Toys (Sylvanite, MT)",2013-01-01,False,3,NaN,3,2016-05-14,3376,116,297663.22,0
2,3,"Tailspin Toys (Peeples Valley, AZ)",2013-01-01,False,3,NaN,3,2016-05-30,3360,126,399699.43,0
3,4,"Tailspin Toys (Medicine Lodge, KS)",2013-01-01,False,3,NaN,3,2016-04-28,3392,102,388743.15,0
4,5,"Tailspin Toys (Gasport, NY)",2013-01-01,False,3,NaN,3,2016-05-28,3362,118,334104.18,0


In [9]:
df_churn = pd.read_excel("Ecommerce.xlsx", sheet_name="Hoja1")

In [10]:
df_churn.info 

<bound method DataFrame.info of      CustomerID                        CustomerName AccountOpenedDate  \
0             1         Tailspin Toys (Head Office)        2013-01-01   
1             2       Tailspin Toys (Sylvanite, MT)        2013-01-01   
2             3  Tailspin Toys (Peeples Valley, AZ)        2013-01-01   
3             4  Tailspin Toys (Medicine Lodge, KS)        2013-01-01   
4             5         Tailspin Toys (Gasport, NY)        2013-01-01   
..          ...                                 ...               ...   
658        1057                     Ganesh Majumdar        2016-01-09   
659        1058                      Jaroslav Fisar        2016-02-01   
660        1059                     Jibek Juniskyzy        2016-03-13   
661        1060                     Anand Mudaliyar        2016-04-23   
662        1061                        Agrita Abele        2016-05-07   

     IsOnCreditHold  CustomerCategoryID  CreditLimit  DeliveryMethodID  \
0                

In [11]:
df_churn.isnull().sum()

CustomerID                 0
CustomerName               0
AccountOpenedDate          0
IsOnCreditHold             0
CustomerCategoryID         0
CreditLimit              402
DeliveryMethodID           0
UltimaCompra               0
DiasDesdeUltimaCompra      0
NumFacturas                0
TotalComprado              0
ClienteEnRiesgo            0
dtype: int64

### Preprocesamiento 

In [13]:
df_churn["AccountOpenedDate"] = pd.to_datetime(df_churn["AccountOpenedDate"], errors="coerce")

In [14]:
# Años como cliente
from datetime import datetime

hoy = pd.Timestamp.today()
df_churn["AñosComoCliente"] = (hoy - df_churn["AccountOpenedDate"]).dt.days


In [15]:
import numpy as np 
# Crear la variable ticket promedio
df_churn["TicketPromedio"] = df_churn["TotalComprado"] / df_churn["NumFacturas"] 

In [16]:
# Reemplazar errores (por ejemplo: division entre cero ó nulos)
df_churn["TicketPromedio"].replace([np.inf, - np.inf], 0 , inplace=True)
df_churn["TicketPromedio"].fillna(0,inplace=True)

In [17]:
#Variables predictoras
features = [
    "NumFacturas",
    "TotalComprado",
    "TicketPromedio",
    "AñosComoCliente",
    "IsOnCreditHold",
    "CreditLimit"
]

In [18]:
# Variable Objetivo
target = "ClienteEnRiesgo"

In [19]:
# Pasar a X, y
X = df_churn[features]
y = df_churn[target]

In [20]:
# Ver cuántos casos de cada clase hay
df_churn["ClienteEnRiesgo"].value_counts()

ClienteEnRiesgo
0    642
1     21
Name: count, dtype: int64

### Clase 1 (Cliente en riesgo)
### Clase 0 (Cliente que no esta en riesgo)

## Crear el modelo 

In [23]:
from sklearn.model_selection import train_test_split
# División 80% entrenamiento, 20% prueba
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [24]:
X

,NumFacturas,TotalComprado,TicketPromedio,AñosComoCliente,IsOnCreditHold,CreditLimit
0,123,397101.76,3228.469593,4605,0,NaN
1,116,297663.22,2566.062241,4605,0,NaN
2,126,399699.43,3172.217698,4605,0,NaN
3,102,388743.15,3811.207353,4605,0,NaN
4,118,334104.18,2831.391356,4605,0,NaN
...,...,...,...,...,...,...
658,18,60639.75,3368.875000,3502,0,1600.0
659,14,70569.28,5040.662857,3479,0,1900.0
660,8,17170.54,2146.317500,3438,0,1800.0
661,4,9412.26,2353.065000,3397,0,1100.0


In [25]:
from sklearn.ensemble import RandomForestClassifier
# Crear y entrenar modelo 
model = RandomForestClassifier(n_estimators = 100, random_state=42)
model.fit(X_train,y_train)

RandomForestClassifier(random_state=42)

In [26]:
X_test

,NumFacturas,TotalComprado,TicketPromedio,AñosComoCliente,IsOnCreditHold,CreditLimit
327,124,406475.88,3278.031290,4605,0,NaN
579,118,364417.19,3088.281271,4605,0,3465.0
513,109,361823.53,3319.481927,4605,0,2600.0
362,118,379137.66,3213.031017,4605,0,NaN
265,119,349389.84,2936.049076,4605,0,NaN
...,...,...,...,...,...,...
465,103,288025.84,2796.367379,4605,0,1300.0
533,97,312417.48,3220.798763,4605,0,1800.0
431,122,392607.04,3218.090492,4605,0,2000.0
248,129,371210.17,2877.598217,4605,0,NaN


In [27]:
from sklearn.metrics import classification_report, confusion_matrix
#Evaluar con datos que no conoce el modelo
y_pred = model.predict(X_test)

In [28]:
# Matriz de confusion
print(confusion_matrix(y_test,y_pred))

[[129   0]
 [  0   4]]


In [29]:
# Repprte de metricas
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       129
           1       1.00      1.00      1.00         4

    accuracy                           1.00       133
   macro avg       1.00      1.00      1.00       133
weighted avg       1.00      1.00      1.00       133



## Validacion cruzada Para RandomForestClassifier

In [31]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
#Validación cruzada estratificada (respesta proporcion de clases)
cv= StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

scores = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X,y,cv=cv, scoring='f1')

print("F1 - score promedio:", scores.mean())
print("F1 - score Individual:", scores)

F1 - score promedio: 0.9777777777777779
F1 - score Individual: [0.93333333 1.         1.        ]


## Entrenar usando el modelo xgboost

In [33]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric='logloss', random_state=42)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print("--- XGBoost---")
print(confusion_matrix(y_test, y_pred_xgb))

--- XGBoost---
[[129   0]
 [  0   4]]


## Validacion cruzada para XGBBoost


In [35]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

scores_xgb = cross_val_score(xgb_model,X,y, cv=cv, scoring="f1")

print("--- XGBoots validacion cruzada---")
print("F1 - score promedio:", scores_xgb.mean())
print("F1 - score Individual:", scores_xgb)

--- XGBoots validacion cruzada---
F1 - score promedio: 0.9555555555555556
F1 - score Individual: [0.93333333 0.93333333 1.        ]


### Random Forest se basa en la diversidad de muchos árboles entrendados en parelo.

### XGBoost construye arboles uno a uno, cada uno mejorando el aterior.

## Comparar  con regresion logistica 

In [38]:
from sklearn.impute import SimpleImputer

#Crear imputador con la media
imputer = SimpleImputer(strategy="mean")

#Ajustar valores imputados
X_train_imputed = imputer.fit_transform(X_train)
x_test_imputed = imputer.fit_transform(X_test)

In [39]:
from sklearn.linear_model import LogisticRegression
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_imputed, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [40]:
y_pred_log = log_model.predict(x_test_imputed)
print(confusion_matrix(y_test,y_pred_log))
print(classification_report(y_test,y_pred_log))

[[129   0]
 [  0   4]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       129
           1       1.00      1.00      1.00         4

    accuracy                           1.00       133
   macro avg       1.00      1.00      1.00       133
weighted avg       1.00      1.00      1.00       133



In [41]:
from sklearn.pipeline import make_pipeline
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#Crear pipeline: imputador + modele
log_pipeline = make_pipeline(
    SimpleImputer(strategy="mean"), # Imputa NaNs con la media
    LogisticRegression(max_iter=1000, random_state=42)
)

# Validacion cruzada
scores_log = cross_val_score(log_pipeline, X,y, cv=cv, scoring='f1')

print("Regresion Logistica")
print("f1 score:",scores_log.mean())
print("f1 score individual:", scores_log)

Regresion Logistica
f1 score: 1.0
f1 score individual: [1. 1. 1. 1. 1.]
